# Ridge and lasso: keeping models simple
**Machine Learning · Week 2 (Regression and gradient descent) · Day 5**

Lesson page: https://sara-aminpour.com/tutorials/ml/w2d5.html

A flexible model can thread every training point and still be wrong everywhere else.
**Regularization** adds a penalty for big weights: a leash on the model's complexity.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso

rng = np.random.default_rng(0)
def truth(x):
    return np.sin(2 * np.pi * x)

x_train = np.sort(rng.uniform(0, 1, 15))
y_train = truth(x_train) + rng.normal(0, 0.25, 15)
x_val = rng.uniform(0, 1, 200)                       # fresh data for validation
y_val = truth(x_val) + rng.normal(0, 0.25, 200)
xs = np.linspace(0, 1, 300)

def mse(model, x, y):
    return np.mean((model.predict(x.reshape(-1, 1)) - y) ** 2)

def poly_model(regressor, degree=14):
    """Polynomial features (standardized), then a linear model on top."""
    return make_pipeline(PolynomialFeatures(degree, include_bias=False), StandardScaler(), regressor)

wild = poly_model(LinearRegression()).fit(x_train.reshape(-1, 1), y_train)
print(f"degree 14, no penalty: training MSE {mse(wild, x_train, y_train):.4f}, validation MSE {mse(wild, x_val, y_val):.1f}")
print("largest weight:", np.abs(wild[-1].coef_).max().round(0))

## Put a leash on it: ridge regression
Ridge adds $\lambda \sum_k w_k^2$ to the loss (scikit-learn calls $\lambda$ `alpha`).
Same degree-14 model, three leash strengths:

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11, 3.4), sharey=True, layout="constrained")
for ax, alpha in zip(axes, [1e-6, 1e-2, 10]):
    model = poly_model(Ridge(alpha=alpha)).fit(x_train.reshape(-1, 1), y_train)
    ax.plot(xs, truth(xs), "--", color="#8a85a3", lw=1.5)
    ax.plot(xs, model.predict(xs.reshape(-1, 1)), color="#17a98c", lw=2.5)
    ax.scatter(x_train, y_train, color="#ff8a50", zorder=3)
    ax.set_ylim(-2, 2)
    ax.set_title(f"ridge, alpha = {alpha:g}\nvalidation MSE {mse(model, x_val, y_val):.3f}")
plt.show()

## Ridge shrinks, lasso zeroes
Lasso uses $\lambda \sum_k |w_k|$ instead. It pushes many weights to exactly zero.

In [ ]:
fits = {
    "no penalty": LinearRegression(),
    "ridge (L2)": Ridge(alpha=1e-2),
    "lasso (L1)": Lasso(alpha=1e-3, max_iter=200_000),
}
fig, ax = plt.subplots(figsize=(7.5, 3.2), layout="constrained")
width = 0.27
for i, (name, reg) in enumerate(fits.items()):
    m = poly_model(reg).fit(x_train.reshape(-1, 1), y_train)
    w = np.abs(m[-1].coef_)
    ax.bar(np.arange(1, 15) + (i - 1) * width, np.maximum(w, 1e-4), width, label=f"{name}: {np.sum(w > 1e-6)} non-zero")
ax.set_yscale("log")
ax.set_xlabel("weight for x^k, k =")
ax.set_ylabel("|weight| (log scale)")
ax.legend()
plt.show()

## Choosing λ by cross-validation
The leash strength is a hyperparameter, so we choose it like $k$ in week 1: with cross-validation
on the training data, never with the test set. `GridSearchCV` tries every value for us.

In [ ]:
from sklearn.model_selection import GridSearchCV

alphas = np.logspace(-8, 2, 21)
search = GridSearchCV(poly_model(Ridge()), {"ridge__alpha": alphas}, cv=5, scoring="neg_mean_squared_error")
search.fit(x_train.reshape(-1, 1), y_train)
best_alpha = search.best_params_["ridge__alpha"]
print(f"best alpha by 5-fold CV: {best_alpha:.1e}")
print(f"its validation MSE on fresh data: {mse(search.best_estimator_, x_val, y_val):.3f}")

plt.figure(figsize=(6.2, 3.2))
plt.plot(alphas, -search.cv_results_["mean_test_score"], "o-", color="#7b5cfa")
plt.axvline(best_alpha, color="#f0ac2e", ls="--")
plt.xscale("log")
plt.yscale("log")
plt.xlabel("alpha (leash strength)")
plt.ylabel("cross-validated MSE")
plt.title("Too loose on the left, too tight on the right")
plt.show()

## Your turn: ridge in PyTorch
In PyTorch, ridge is the optimizer's `weight_decay` argument. Train the degree-12 polynomial with and
without it and compare the validation error and the largest weight.

In [ ]:
import torch

def features(x, degree=12):
    F = np.vander(2 * x - 1, degree + 1, increasing=True)[:, 1:]       # x mapped to [-1, 1], then powers
    return torch.tensor(F, dtype=torch.float32)

Xtr, Ytr = features(x_train), torch.tensor(y_train, dtype=torch.float32).reshape(-1, 1)
Xva, Yva = features(x_val), torch.tensor(y_val, dtype=torch.float32).reshape(-1, 1)

def train(weight_decay):
    torch.manual_seed(0)
    model = torch.nn.Linear(12, 1)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.05)   # TODO: pass weight_decay=weight_decay
    for step in range(3000):
        optimizer.zero_grad()
        torch.nn.functional.mse_loss(model(Xtr), Ytr).backward()
        optimizer.step()
    val = torch.nn.functional.mse_loss(model(Xva), Yva).item()
    return val, model.weight.abs().max().item()

for wd in [0.0, 1e-2, 1e-1, 1.0]:
    val, biggest = train(wd)
    print(f"weight_decay = {wd:g}: validation MSE {val:.3f}, largest weight {biggest:.1f}")

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
def train(weight_decay):
    torch.manual_seed(0)
    model = torch.nn.Linear(12, 1)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.05, weight_decay=weight_decay)
    for step in range(3000):
        optimizer.zero_grad()
        torch.nn.functional.mse_loss(model(Xtr), Ytr).backward()
        optimizer.step()
    val = torch.nn.functional.mse_loss(model(Xva), Yva).item()
    return val, model.weight.abs().max().item()

for wd in [0.0, 1e-2, 1e-1, 1.0]:
    val, biggest = train(wd)
    print(f"weight_decay = {wd:g}: validation MSE {val:.3f}, largest weight {biggest:.1f}")
print("A little weight decay keeps the weights small and the curve calm. Too much and it underfits.")